This notebook reads all distinct vessel.id values from your Bronze events (port_visit, gap, fishing, loitering, encounter), compares them against the vessels already stored in gfw/vessels/ so it never fetches the same vessel twice, calls the Global Fishing Watch v3 /vessels API in batches of 50 IDs, and saves new vessel profiles to gfw/vessels/load_type=incremental/ingest_date=<YYYY-MM-DD>/.

In [0]:
import json
import time
from datetime import datetime, timezone
import requests
from pyspark.sql import functions as F

# ── Configuration ────────────────────────────────────────────────────────────
SA_NAME = "stmaritimeks"
BRONZE_ROOT = f"abfss://bronze@{SA_NAME}.dfs.core.windows.net"
EVENTS_ROOT = f"{BRONZE_ROOT}/gfw/events"
VESSELS_ROOT = f"{BRONZE_ROOT}/gfw/vessels"
BASE_URL = "https://gateway.api.globalfishingwatch.org/v3"

# Update KV_SCOPE tomorrow once Member 1 confirms the exact scope name
KV_SCOPE = "maritime-kv"
API_TOKEN = dbutils.secrets.get(scope=KV_SCOPE, key="gfw-api-token")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

VESSEL_DATASETS = [
    "public-global-vessel-identity:latest",
]
BATCH_SIZE = 50


def collect_missing_vessel_ids():
    # 1. Read all distinct vessel.id values from all 5 event types in Bronze
    event_types = ["port_visit", "gap", "fishing", "loitering", "encounter"]
    event_vessel_dfs = []
    for et in event_types:
        path = f"{EVENTS_ROOT}/{et}/load_type=*/ingest_date=*/*.jsonl"
        try:
            df = spark.read.json(path).select(F.col("vessel.id").alias("vessel_id"))
            event_vessel_dfs.append(df)
        except Exception as exc:
            print(f"[WARN] Could not read {et} events: {exc}")

    if not event_vessel_dfs:
        return []

    all_event_vessels = event_vessel_dfs[0]
    for df in event_vessel_dfs[1:]:
        all_event_vessels = all_event_vessels.unionByName(df)

    distinct_event_ids = (
        all_event_vessels
        .filter(F.col("vessel_id").isNotNull() & (F.trim(F.col("vessel_id")) != ""))
        .distinct()
    )

    # 2. Read already-enriched vessel IDs from Bronze vessels folder
    existing_path = f"{VESSELS_ROOT}/load_type=*/ingest_date=*/*.jsonl"
    try:
        existing_df = (
            spark.read.json(existing_path)
            .select(F.explode_outer("selfReportedInfo").alias("sri"))
            .select(F.col("sri.id").alias("vessel_id"))
            .filter(F.col("vessel_id").isNotNull())
            .distinct()
        )
        missing_df = distinct_event_ids.join(existing_df, on="vessel_id", how="left_anti")
    except Exception:
        # No existing vessel files yet
        missing_df = distinct_event_ids

    missing_ids = [row["vessel_id"] for row in missing_df.collect()]
    return missing_ids


def fetch_vessel_batch_with_retry(vessel_ids: list):
    url = f"{BASE_URL}/vessels"
    params = [("datasets[0]", VESSEL_DATASETS[0])]
    for idx, vid in enumerate(vessel_ids):
        params.append((f"ids[{idx}]", vid))

    retries_used = 0
    for attempt in range(1, 6):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if resp.status_code == 200:
            data = resp.json()
            return data.get("entries", []), retries_used
        if resp.status_code in (429, 500, 502, 503, 504):
            retries_used += 1
            wait_sec = 2 ** attempt
            print(f"  [HTTP {resp.status_code}] Retrying in {wait_sec}s (attempt {attempt}/5)...")
            time.sleep(wait_sec)
        else:
            resp.raise_for_status()

    raise RuntimeError("Failed after 5 retries fetching vessel batch")


def run_vessel_enrichment():
    ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    run_id = f"vessels_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

    missing_ids = collect_missing_vessel_ids()
    print(f"Found {len(missing_ids)} new unenriched vessel IDs to fetch.")

    if not missing_ids:
        print("All event vessels are already enriched in Bronze. Nothing to fetch.")
        return

    manifest_records = []
    total_batches = (len(missing_ids) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num in range(1, total_batches + 1):
        start_idx = (batch_num - 1) * BATCH_SIZE
        batch_ids = missing_ids[start_idx : start_idx + BATCH_SIZE]
        started_at = datetime.now(timezone.utc).isoformat()
        target_path = (
            f"{VESSELS_ROOT}/load_type=incremental/"
            f"ingest_date={ingest_date}/batch_{batch_num:04d}.jsonl"
        )

        try:
            entries, retries = fetch_vessel_batch_with_retry(batch_ids)
            body = "\n".join(json.dumps(r) for r in entries)
            dbutils.fs.put(target_path, body, True)
            status = "SUCCESS"
            err_msg = None
            print(f"[OK] Batch {batch_num}/{total_batches}: fetched {len(entries)} vessels -> {target_path}")
        except Exception as exc:
            entries = []
            retries = 5
            status = "FAILED"
            err_msg = str(exc)
            print(f"[ERR] Batch {batch_num}/{total_batches}: {err_msg}")

        manifest_records.append({
            "run_id": run_id,
            "entity": "vessels",
            "dataset": VESSEL_DATASETS[0],
            "region": "world",
            "window_start": ingest_date,
            "window_end": ingest_date,
            "load_type": "incremental",
            "source": "gfw_v3_vessels_api",
            "rows_received": len(entries),
            "file_path": target_path,
            "status": status,
            "retry_count": retries,
            "error_message": err_msg,
            "started_at_utc": started_at,
            "finished_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_url": f"{BASE_URL}/vessels",
            "api_version": "v3",
        })
        time.sleep(0.25)

    manifest_path = f"{BRONZE_ROOT}/_manifest/ingestion_manifest/{run_id}.jsonl"
    dbutils.fs.put(manifest_path, "\n".join(json.dumps(r) for r in manifest_records), True)
    print(f"\nVessel enrichment manifest written to {manifest_path}")


if __name__ == "__main__":
    run_vessel_enrichment()